# MassGAT: Results Visualization
This notebook provides an interactive tool to explore MassGAT inference results.

**Prerequisites:** The MassGAT pipeline (IonShed → Graph Construction → Inference) must have already been executed.

## Setup: Configure Paths
Define the project workspace and point to the results from a previous pipeline run.

In [1]:
import os
from pathlib import Path

# Define the project workspace and directories
os.environ["PROJ_NAME"] = "test"  # Change this to your project name
DOC_DIR = Path(os.getcwd())
BASE_DIR = DOC_DIR.parent
DATA_DIR = BASE_DIR / "data" / os.environ["PROJ_NAME"]
GRAPHS_DIR = DATA_DIR / "graphs"
RESULTS_DIR = DATA_DIR / "results"

# Define ionization mode (must match the pipeline run)
IONIZATION_MODE = "positive"  # "positive" or "negative"

print(f"Data directory: {DATA_DIR}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Graphs directory: {GRAPHS_DIR}")
print(f"Ionization mode: {IONIZATION_MODE}")

Data directory: \\fouet\spi\scidospi\08_Team\PinartPaulHenri\massgat\data\test
Results directory: \\fouet\spi\scidospi\08_Team\PinartPaulHenri\massgat\data\test\results
Graphs directory: \\fouet\spi\scidospi\08_Team\PinartPaulHenri\massgat\data\test\graphs
Ionization mode: positive


---
## Interactive Results Visualization
Use the interface below to search for a specific chemical target in your processed graphs. Enter your expected **m/z** and **Retention Time (min)** to locate the component, extract the pseudo-spectrum, and visualize MassGAT's predictions.

In [ ]:
import re, glob
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd, torch, networkx as nx

from src.utils.visualizations import create_nx_graph, visualize_component

# Setup directories for visualizations
VIS_DIR = DATA_DIR / "visualizations"
VIS_DIR.mkdir(parents=True, exist_ok=True)

display(HTML("""<style>
.search-card{background:#fff;border:1px solid #e5e7eb;border-radius:12px;padding:14px 16px 10px;
             box-shadow:0 1px 3px rgba(0,0,0,.06);}
.search-card .lbl{font:600 11px/1.8 system-ui,sans-serif;color:#6b7280;
                  letter-spacing:.05em;text-transform:uppercase;margin-top:6px;}
.search-card .widget-dropdown>select,
.search-card .widget-text>input,
.search-card .widget-numeric-text>input{border-radius:8px!important;}
.search-card .widget-button{border-radius:8px!important;}
</style>"""))

ST = {'description_width': 'initial'}

# the user can set the height
PLOT_CONFIG = {'toImageButtonOptions': {'format': 'svg', 'filename': 'interactive_pseudospectrum'},
               'displaylogo': False, 'responsive': True}


def L(width='250px'):
    """A FRESH Layout per widget: sharing one instance makes hide/show leak."""
    return widgets.Layout(width=width)


def row(*children):
    """HBox that wraps instead of overflowing on a narrow screen."""
    return widgets.HBox(list(children), layout=widgets.Layout(flex_flow='row wrap'))


def show(w, on):
    """ipywidgets has no `visible` trait since v7 - use layout.display."""
    w.layout.display = None if on else 'none'


def header(text):
    return widgets.HTML(f"<div class='lbl'>{text}</div>")


# Widgets
tsv_files = sorted(Path(p).name for p in glob.glob(str(RESULTS_DIR / "*_peaks.tsv")))

w_file = widgets.Dropdown(options=tsv_files, description='Results TSV:', style=ST, layout=L())
w_graph = widgets.Dropdown(options=[], description='Graph (.pt):', style=ST, layout=L('320px'))

w_mz = widgets.FloatText(value=152.0556, description='m/z:', step=0.0001, style=ST, layout=L('200px'))
w_tol_unit = widgets.Dropdown(options=['ppm', 'absolute m/z'], description='Tol. unit:', style=ST, layout=L('210px'))
w_ppm = widgets.FloatText(value=5.0, description='± ppm:', step=1.0, style=ST, layout=L('160px'))
w_mz_abs = widgets.FloatText(value=0.005, description='± m/z:', step=0.001, style=ST, layout=L('160px'))

w_use_rt = widgets.Checkbox(value=True, description='Filter on RT', indent=False, layout=L('120px'))
w_rt = widgets.FloatText(value=4.47, description='RT (min):', step=0.01, style=ST, layout=L('200px'))
w_rt_tol = widgets.FloatText(value=5.0, description='± RT (s):', step=1.0, style=ST, layout=L('160px'))

w_btn = widgets.Button(description='Search', button_style='primary', icon='search', layout=L('130px'))
w_match = widgets.Dropdown(options=[], description='Match:', style=ST, disabled=True, layout=L('530px'))
w_height = widgets.IntSlider(value=560, min=350, max=1000, step=20,
                             description='Fig. height (px):', continuous_update=False,
                             readout_format='d', style=ST, layout=L('360px'))
w_title = widgets.Text(value='Guanine', description='Save title:', style=ST, layout=L())
w_save = widgets.Button(description='Save HTML', button_style='success', icon='save',
                        disabled=True, layout=L('130px'))

out_status = widgets.Output()
out_plot = widgets.Output()

state = {'key': None, 'graph': None, 'G': None, 'node_probs': None, 'node': None}
last_args = {}


def log(msg, clear=True):
    with out_status:
        if clear:
            clear_output(wait=True)
        print(msg)


# Reactive UI
show(w_mz_abs, False)


def toggle_tol_unit(c):
    show(w_ppm, c['new'] == 'ppm')
    show(w_mz_abs, c['new'] != 'ppm')


def toggle_rt(c):
    w_rt.disabled = w_rt_tol.disabled = not c['new']


w_tol_unit.observe(toggle_tol_unit, names='value')
w_use_rt.observe(toggle_rt, names='value')


def update_graph_options(*_):
    if not w_file.value:
        w_graph.options = []
        return
    try:
        col = pd.read_csv(RESULTS_DIR / w_file.value, sep='\t', usecols=['file_name'])['file_name']
        w_graph.options = sorted(col.dropna().unique().tolist())
    except Exception as e:
        w_graph.options = []
        log(f"Could not read {w_file.value}: {e}")


w_file.observe(update_graph_options, names='value')
update_graph_options()


# Data loading (cached)
def load_data():
    """Load TSV + graph only when the selection actually changed."""
    key = (w_file.value, w_graph.value)
    if state['key'] == key:
        return
    df = pd.read_csv(RESULTS_DIR / w_file.value, sep='\t')
    state['node_probs'] = dict(zip(zip(df['file_name'], df['roi_id']), df['node_prob']))
    state['graph'] = torch.load(GRAPHS_DIR / w_graph.value, map_location='cpu')
    state['G'] = create_nx_graph(state['graph'], ionization_mode=IONIZATION_MODE)
    state['key'] = key


def node_prob(i):
    """node index -> probability, via the roi stored by create_nx_graph."""
    if state['G'] is None or i not in state['G']:
        return None
    roi = state['G'].nodes[i]['roi']
    p = state['node_probs'].get((w_graph.value, roi))
    if p is None:          # roi_values may be float while roi_id is int
        p = state['node_probs'].get((w_graph.value, int(roi)))
    return p


# Search
def on_search(_):
    w_save.disabled = w_match.disabled = True
    w_match.options = []
    w_btn.disabled, w_btn.description = True, 'Searching…'
    try:
        if not w_file.value or not w_graph.value:
            return log("Select a results TSV and a graph first.")

        log("Loading data…")
        load_data()

        mz_feat = state['graph']["node_features"][:, 0]
        rt_feat = state['graph']["node_features"][:, 1]  # seconds

        tol = w_mz.value * w_ppm.value / 1e6 if w_tol_unit.value == 'ppm' else w_mz_abs.value
        keep = torch.abs(mz_feat - w_mz.value) <= tol
        if w_use_rt.value:
            keep &= torch.abs(rt_feat - w_rt.value * 60.0) <= w_rt_tol.value

        idxs = torch.where(keep)[0].tolist()
        if not idxs:
            return log("No match. Try widening the m/z or RT tolerance.")

        idxs.sort(key=lambda i: abs(mz_feat[i].item() - w_mz.value))   # closest first
        options = []
        for i in idxs:
            mz_i, rt_i = mz_feat[i].item(), rt_feat[i].item() / 60.0
            err = (mz_i - w_mz.value) / w_mz.value * 1e6
            p = node_prob(i)
            p_txt = f"p={p:.2f}" if p is not None else "p=n/a"
            options.append((f"Node {i}  ·  m/z {mz_i:.4f} ({err:+.1f} ppm)"
                            f"  ·  RT {rt_i:.2f} min  ·  {p_txt}", i))

        w_match.options = options
        w_match.disabled = False
        log(f"{len(options)} match(es) — closest in mass listed first.")


        first = options[0][1]
        if w_match.value == first:
            render(first)             
        else:
            w_match.value = first     
    except Exception as e:
        log(f"{type(e).__name__}: {e}")
    finally:
        w_btn.disabled, w_btn.description = False, 'Search'


# Render
def render(node):
    state['node'] = node
    if node is None or state['G'] is None:
        return
    w_save.disabled = True
    if node not in state['G']:
        return log(f"Node {node} is not present in the graph.")

    component = nx.node_connected_component(state['G'], node)
    p = node_prob(node)
    log(f"Node {node} — component of {len(component)} peak(s)"
        + (f", p={p:.2f}" if p is not None else "") + ".")

    last_args.clear()
    last_args.update(dict(G=state['G'], nodes=component, highlight_nodes=[node],
                          node_probs=state['node_probs'], graph_name=w_graph.value,
                          ionization_mode=IONIZATION_MODE, remove_low_prob=True,
                          use_prob_styling=True, show_profiles=True,
                          graph=state['graph'], style="presentation"))
    with out_plot:
        clear_output(wait=True)
        fig = visualize_component(**last_args, show=False)
        if fig is None:
            return log("Nothing to display: every node is below the probability threshold.")
        fig.update_layout(height=w_height.value)             
        fig.show(config=PLOT_CONFIG)
    w_save.disabled = False


w_match.observe(lambda c: render(c['new']), names='value')
w_height.observe(lambda c: render(state['node']), names='value')


# Save
def on_save(_):
    if not last_args:
        return log("Nothing to save yet.")
    base = w_graph.value.replace("graph_", "").replace(".pt", "")
    title = re.sub(r'[^A-Za-z0-9._-]+', '_', w_title.value.strip()) or "component"
    path = VIS_DIR / f"{base}_{title}_mz{w_mz.value:.4f}_node{state['node']}.html"
    visualize_component(**last_args, save_html=str(path), show=False)
    log(f"Saved: {path}")


w_btn.on_click(on_search)
w_save.on_click(on_save)

ui = widgets.VBox([
    header("Source"),
    row(w_file, w_graph),
    header("Target"),
    row(w_mz, w_tol_unit, w_ppm, w_mz_abs),
    row(w_use_rt, w_rt, w_rt_tol),
    header("Result"),
    row(w_btn, w_match, w_height),
    row(w_title, w_save),
])
ui.add_class('search-card')
display(ui, out_status, out_plot)

Output()

Output()